# MCQ Generation — Step 1: File Loading and Preprocessing

Loads all `_v1_chunks.json` files into a unified corpus, merges figure captions into
parent chunks, filters out reference/stub/figure-only chunks, and builds an enriched
context string (section path prepended) for each surviving chunk.

In [1]:
import glob
import json
import re
from dataclasses import dataclass, field
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CHUNKS_GLOB = str(REPO_ROOT / "output" / "*" / "auto" / "*_v1_chunks.json")

MIN_CHUNK_CHARS = 100
EXCLUDED_SECTION_KEYWORDS = ("REFERENCES", "BIBLIOGRAPHY")
FIGURE_PLACEHOLDER_RE = re.compile(r">\s*\[FIGURE:[^\]]+\]")

## Load raw chunk files

Each `_v1_chunks.json` is `{doc_id, source, chunks, figures}`. `figures[i].referenced_from`
points at the `chunk_id` it belongs to — that's the join key for the caption merge.

In [2]:
@dataclass
class Chunk:
    uid: str
    doc_id: str
    chunk_id: str
    section_path: list[str]
    text: str
    context: str = field(default="")


def load_chunk_files(pattern: str = CHUNKS_GLOB) -> list[dict]:
    paths = sorted(glob.glob(pattern))
    assert paths, f"no chunk files matched {pattern!r}"
    docs = [json.load(open(p, encoding="utf-8")) for p in paths]
    print(f"loaded {len(docs)} doc(s) from {len(paths)} file(s)")
    return docs


raw_docs = load_chunk_files()

loaded 16 doc(s) from 16 file(s)


## Merge figure captions into parent chunks

Note: in this corpus the captioning pipeline already inlines `[FIGURE:fig_id] <caption>`
into chunk text (see `src/ingestion/upload_visuals.py` / `merge_by_page.py`). This step
is still applied defensively via `referenced_from` so the pipeline is correct even against
chunk files where captions were *not* pre-merged upstream — appending is a no-op when the
caption text is already a substring of the chunk body.

In [3]:
def merge_captions(doc: dict) -> dict[str, str]:
    """Return chunk_id -> text, with any not-yet-inlined figure captions appended."""
    captions_by_chunk: dict[str, list[str]] = {}
    for fig in doc.get("figures", []):
        parent = fig.get("referenced_from")
        caption = fig.get("caption", "").strip()
        if parent and caption:
            captions_by_chunk.setdefault(parent, []).append(caption)

    merged: dict[str, str] = {}
    for chunk in doc["chunks"]:
        text = chunk["text"]
        for caption in captions_by_chunk.get(chunk["chunk_id"], []):
            if caption not in text:
                text = f"{text}\n\n{caption}"
        merged[chunk["chunk_id"]] = text
    return merged

## Filter unusable chunks

Drops: reference/bibliography sections, sub-100-char stub chunks, and chunks that are
purely a figure placeholder with no surrounding prose.

In [4]:
def is_reference_section(section_path: list[str]) -> bool:
    joined = " ".join(section_path).upper()
    return any(kw in joined for kw in EXCLUDED_SECTION_KEYWORDS)


def is_figure_placeholder_only(text: str) -> bool:
    return not FIGURE_PLACEHOLDER_RE.sub("", text).strip()


def should_keep(section_path: list[str], text: str) -> bool:
    if is_reference_section(section_path):
        return False
    if len(text) < MIN_CHUNK_CHARS:
        return False
    if is_figure_placeholder_only(text):
        return False
    return True

## Build enriched context (section path as topic anchor)

The most specific heading — the last element of `section_path` — is prepended so the
LLM knows the governing concept before reading bullet-fragment body text.

In [5]:
def build_context(section_path: list[str], text: str) -> str:
    topic = section_path[-1] if section_path else ""
    return f"{topic}\n\n{text}" if topic else text

## Assemble the unified corpus

In [6]:
def build_corpus(raw_docs: list[dict]) -> list[Chunk]:
    corpus: list[Chunk] = []
    dropped = 0
    for doc in raw_docs:
        doc_id = doc["doc_id"]
        merged_text_by_id = merge_captions(doc)
        for chunk in doc["chunks"]:
            chunk_id = chunk["chunk_id"]
            section_path = chunk["section_path"]
            text = merged_text_by_id[chunk_id]

            if not should_keep(section_path, text):
                dropped += 1
                continue

            corpus.append(
                Chunk(
                    uid=f"{doc_id}::{chunk_id}",
                    doc_id=doc_id,
                    chunk_id=chunk_id,
                    section_path=section_path,
                    text=text,
                    context=build_context(section_path, text),
                )
            )
    print(f"kept {len(corpus)} chunk(s), dropped {dropped}")
    return corpus


corpus = build_corpus(raw_docs)

kept 413 chunk(s), dropped 27


In [7]:
# Sanity check — inspect a few surviving chunks
for chunk in corpus[:3]:
    print(chunk.uid)
    print(chunk.context[:300])
    print("---")

2025 04 21 ENDOCRINE.1-25 - dr. Imam.pdf_origin::c002
FUNCTIONS OF HORMONE

A given hormone usually affects only a limited number of cells, called target cells and responds to a hormone control because it has specific receptors of the hormone.

Hormones are synthesized, exist in a biologically active state for a time and then degrade, metabolized or de
---
2025 04 21 ENDOCRINE.1-25 - dr. Imam.pdf_origin::c003
FEEDBACK CONTROL OF HORMONE PRODUCTION

> [FIGURE:p14_b1] The image is a flowchart divided into three main sections, each labeled with "Hormonal Feedback," "Neural Feedback," and "Direct Feedback." Each section contains interconnected nodes with arrows indicating the direction of the feedback loop.

---
2025 04 21 ENDOCRINE.1-25 - dr. Imam.pdf_origin::c004
ENDOCRINOLOGY

> [FIGURE:p18_b1] The image is divided into three main sections, each labeled at the top:

1. **Protein and Peptide Hormones**:
   - At the top left, there is a label "Protein and Peptide Hormones."
   - Below thi

# Step 2 — Triple Extraction from Each Chunk

For each enriched chunk, a single Qwen3-8B prompt extracts candidate `{subject, relation, object}`
triples and assigns a difficulty tier per triple, in one pass. Relations are constrained to a
fixed taxonomy so distractor selection later has a closed vocabulary to work with.

Difficulty tiers:
- **1 (recall)** — a direct named relationship between two entities.
- **2 (application)** — involves a mechanism or process.
- **3 (reasoning)** — requires connecting this triple with another fact from the same chunk.

In [8]:
from typing import Literal

from pydantic import BaseModel, Field

RELATION_TAXONOMY = (
    "PRODUCES",
    "REGULATES",
    "INHIBITS",
    "STIMULATES",
    "ACTS_ON",
    "PART_OF",
    "LOCATED_IN",
    "INNERVATES",
    "SUPPLIES",
    "CAUSES",
    "DIAGNOSED_BY",
    "TREATED_BY",
    "PREREQUISITE_OF",
)

Relation = Literal[
    "PRODUCES",
    "REGULATES",
    "INHIBITS",
    "STIMULATES",
    "ACTS_ON",
    "PART_OF",
    "LOCATED_IN",
    "INNERVATES",
    "SUPPLIES",
    "CAUSES",
    "DIAGNOSED_BY",
    "TREATED_BY",
    "PREREQUISITE_OF",
]

DifficultyTier = Literal[1, 2, 3]


class Triple(BaseModel):
    subject: str = Field(min_length=1)
    relation: Relation
    object: str = Field(min_length=1)
    difficulty: DifficultyTier


class TripleExtractionResult(BaseModel):
    triples: list[Triple]

In [ ]:
# Qwen3.5-9B triple extractor: reads a chunk's enriched context and emits a
# constrained-taxonomy relation triple list with per-triple difficulty tier, in one pass.
#
# Loaded lazily and released after use, matching the captioning module's memory-management
# pattern (src/captioning/qwen_vl.py) — same 24GB unified memory budget on MPS, only one
# model resident at a time.
#
# Qwen3.5 is a multimodal checkpoint (AutoModelForMultimodalLM + AutoProcessor rather than
# AutoModelForCausalLM + AutoTokenizer) but runs fine text-only — just omit image content
# from the messages.
#
# enable_thinking=False: this is structured extraction, not open-ended reasoning — thinking
# mode burns tokens producing a chain-of-thought this task doesn't need and complicates
# parsing the JSON back out.

import gc
import json
import logging
import re

import torch
from pydantic import ValidationError
from transformers import AutoModelForMultimodalLM, AutoProcessor

MODEL_PATH = "Qwen/Qwen3.5-9B"

log = logging.getLogger("triple_extractor")

_SYSTEM_PROMPT = f"""You are a medical knowledge graph extractor. Read the given passage and \
extract factual relationship triples between named medical entities (organs, hormones, \
enzymes, structures, diseases, processes, etc).

Each triple must use exactly one relation from this fixed taxonomy — do not invent new \
relations, do not use synonyms, only these:
{", ".join(RELATION_TAXONOMY)}

Never use a passive/inverse form of a relation (e.g. "STIMULATED_BY", "REGULATED_BY", \
"INHIBITED_BY", "SYNTHESIZED_BY", "DEGRADED_BY" are all forbidden). If the natural \
direction of a fact is passive, flip subject and object instead so the active relation \
from the taxonomy applies — e.g. "TSH is stimulated by TRH" becomes \
{{"subject": "TRH", "relation": "STIMULATES", "object": "TSH"}}, not \
{{"subject": "TSH", "relation": "STIMULATED_BY", "object": "TRH"}}.

For each triple, assign a difficulty tier:
- 1 (recall): a direct named relationship between two entities, statable from a single fact.
- 2 (application): involves a mechanism or process, not just a bare fact.
- 3 (reasoning): answering it requires connecting this triple with at least one other fact \
from the same passage.

Only extract triples that are explicitly supported by the passage text. Do not infer outside \
medical knowledge that isn't stated. If no valid triples exist, return an empty list.

Keep subject/object entity names short (a few words) — do not include full descriptive \
clauses in an entity name.

Respond with ONLY a JSON object of the form:
{{"triples": [{{"subject": "...", "relation": "...", "object": "...", "difficulty": 1}}]}}
No prose, no markdown fences, no explanation — JSON only."""

_JSON_BLOCK_RE = re.compile(r"\{.*\}", re.DOTALL)


def get_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def _repair_truncated_json(raw: str) -> str:
    """Best-effort fix for JSON cut off mid-generation (max_new_tokens truncation): drops
    back to the last fully-formed value (closing an unterminated string first if needed),
    then closes whatever braces/brackets are still open — in the correct nesting order —
    so json.loads has a chance at the still-complete triples."""
    text = raw[raw.index("{"):] if "{" in raw else raw

    if text.count('"') % 2 == 1:
        text = text[: text.rindex('"')]

    last_safe = max(text.rfind("}"), text.rfind("]"), text.rfind(","))
    if last_safe == -1:
        raise ValueError("no safe truncation point found")
    text = text[:last_safe] if text[last_safe] == "," else text[: last_safe + 1]

    # walk the (string-aware) delimiter stack to close whatever's still open, innermost first
    stack: list[str] = []
    in_string = False
    escape = False
    for ch in text:
        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
            continue
        if ch == '"':
            in_string = True
        elif ch in "{[":
            stack.append(ch)
        elif ch in "}]":
            stack.pop()

    closers = {"{": "}", "[": "]"}
    return text + "".join(closers[ch] for ch in reversed(stack))


class TripleExtractor:
    def __init__(self, model_path: str = MODEL_PATH):
        self.model_path = model_path
        self.device = get_device()
        self.model = None
        self.processor = None

    def load(self):
        if self.model is not None:
            return
        self.processor = AutoProcessor.from_pretrained(self.model_path)
        self.model = AutoModelForMultimodalLM.from_pretrained(
            self.model_path,
            dtype=torch.bfloat16
        )
        self.model.to(self.device)
        self.model.eval()

    def unload(self):
        self.model = None
        self.processor = None
        gc.collect()
        if torch.backends.mps.is_available():
            torch.mps.empty_cache()
        elif torch.cuda.is_available():
            torch.cuda.empty_cache()

    def extract(self, context: str, max_new_tokens: int = 2048, max_retries: int = 2) -> TripleExtractionResult:
        """Retries only if EVERY triple in a response is invalid (i.e. the model produced
        unusable JSON or nothing worth keeping) — a response with some good and some bad
        triples keeps the good ones rather than discarding the whole batch."""
        assert self.model is not None, "call load() first"
        last_error: Exception | None = None
        for attempt in range(max_retries + 1):
            raw = self._generate(context, max_new_tokens=max_new_tokens)
            try:
                result = self._parse(raw)
            except ValueError as e:
                last_error = e
                log.warning("extraction parse failed (attempt %d/%d): %s", attempt + 1, max_retries + 1, e)
                log.warning("raw output was: %r", raw)
                continue
            if result.triples:
                return result
            last_error = ValueError("no valid triples in response")
            log.warning("extraction yielded 0 valid triples (attempt %d/%d)", attempt + 1, max_retries + 1)
        assert last_error is not None
        raise last_error

    def _parse(self, raw: str) -> TripleExtractionResult:
        match = _JSON_BLOCK_RE.search(raw)
        if not match:
            raise ValueError(f"no JSON object found in model output: {raw[:200]!r}")

        candidate = match.group(0)
        try:
            payload = json.loads(candidate)
        except json.JSONDecodeError:
            payload = json.loads(_repair_truncated_json(candidate))

        raw_triples = payload.get("triples", [])
        triples: list[Triple] = []
        for item in raw_triples:
            try:
                triples.append(Triple.model_validate(item))
            except ValidationError as e:
                log.warning("dropping invalid triple %r: %s", item, e)
        return TripleExtractionResult(triples=triples)

    def _generate(self, context: str, max_new_tokens: int) -> str:
        messages = [
            {"role": "system", "content": _SYSTEM_PROMPT},
            {"role": "user", "content": context},
        ]
        inputs = self.processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            chat_template_kwargs={"enable_thinking": False},
        ).to(self.device)
        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=None,
                top_p=None,
                top_k=None,
            )
        new_tokens = output_ids[0][inputs["input_ids"].shape[1] :]
        return self.processor.decode(new_tokens, skip_special_tokens=True)


extractor = TripleExtractor()
extractor.load()

/Users/michaeleko/Documents/Works/aiml-institute/challenge-2/intelligent-tutoring-system-for-medical-student/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 4 files:   0%|          | 0/4 [01:07<?, ?it/s]


In [ ]:
from dataclasses import dataclass

from tqdm.auto import tqdm


@dataclass
class ExtractedChunk:
    chunk: "Chunk"
    triples: list[Triple]


def extract_triples_for_corpus(corpus: list, extractor: TripleExtractor) -> list[ExtractedChunk]:
    results: list[ExtractedChunk] = []
    failures = 0
    for chunk in tqdm(corpus, desc="extracting triples"):
        try:
            result = extractor.extract(chunk.context)
        except Exception as e:
            failures += 1
            tqdm.write(f"skipping {chunk.uid}: {e}")
            continue
        results.append(ExtractedChunk(chunk=chunk, triples=result.triples))
    print(f"extracted triples for {len(results)}/{len(corpus)} chunk(s), {failures} failure(s)")
    return results

In [ ]:
# Smoke test on a small sample before running the full corpus (413 chunks is slow on
# local MPS inference) — bump SAMPLE_SIZE to None to run everything.
SAMPLE_SIZE = 5
sample = corpus[:SAMPLE_SIZE] if SAMPLE_SIZE else corpus

extracted = extract_triples_for_corpus(sample, extractor)

In [ ]:
# Sanity check — inspect extracted triples
for ec in extracted:
    print(ec.chunk.uid)
    for t in ec.triples:
        print(f"  ({t.subject}) -[{t.relation}]-> ({t.object})  tier={t.difficulty}")
    print("---")